# board_assembly: 64 square predictions -> board state (M4)

Input: a (64, 13) probability array from any square classifier (rows in python-chess
square order a1..h8, columns in `CLASSES` order). See `docs/contracts.md`.
Output: a `BoardPrediction` (FEN, per-square labels/confidence, low-confidence squares,
legality verdict). This is also the "identify the board state from the camera" item of the
full design; legality here is a *plausibility* check on the placement, separate from the
physical placement check (`placement.ipynb`) and the engine comparison (`board_sync.ipynb`).

Milestone: M4 (board assembly + evaluation).

**TODO** (see `docs/M4_TODO.md`)
- [ ] `check_legality`: king counts, piece caps, pawns on back ranks; then
      `board.status()`-based checks (kings adjacent, side not to move in check); handle the
      unknown turn; promotion-aware piece caps (`MAX_PIECES`)
- [ ] `assemble`: argmax, confidence, low-confidence flags, legality, expected-board diff
- [ ] `repair`: swap lowest-confidence squares to their second-best class until legal;
      call it from `assemble` when `legal` is False
- [ ] `one_hot` test helper, then the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

import chess
import numpy as np

%run ./labels.ipynb

## Piece caps and the prediction record

In [ ]:
# Max count per piece letter in a reachable position (8 pawns; promotions ignored here).
# TODO M4: promotions make e.g. 3 queens legal; decide whether to allow
# "pawns + extra pieces <= 8 + original pieces" instead of fixed caps.
MAX_PIECES = {"K": 1, "Q": 9, "R": 10, "B": 10, "N": 10, "P": 8}
MAX_PER_COLOR = 16


@dataclass
class BoardPrediction:
    fen: str                       # piece placement only (board_fen), e.g. "rnbqkbnr/pppppppp/8/..."
    labels: list[str]              # 64 labels, square-index order
    confidence: list[float]        # 64 max-probabilities
    low_confidence: list[str] = field(default_factory=list)  # square names below threshold
    legal: bool = True
    issues: list[str] = field(default_factory=list)          # human-readable problems

## Legality (plausibility) checks

In [ ]:
def check_legality(board: chess.Board) -> list[str]:
    """Basic plausibility checks on piece placement. Empty list = no problems found.

    Issues are human-readable strings, e.g. "white has 2 kings (expected 1)",
    "black has 9 P (max 8)", "pawn on back rank at a8".
    """
    # Design:
    # - Per color: exactly one king; total pieces <= MAX_PER_COLOR; each type <= MAX_PIECES
    #   (board.pieces(piece_type, color)).
    # - No pawns on rank 1 or 8 (board.pawns & (chess.BB_RANK_1 | chess.BB_RANK_8)).
    # - Then board.status() flags (opposite check, too many checkers, kings adjacent). A
    #   board_fen has no turn: try both turns, or take the turn from the game record.
    # Why strings: they go straight into the report / UI; keep them stable so tests can match.
    # TODO: implement
    raise NotImplementedError

## Assembly
Argmax per square, flag squares whose top probability is below the threshold
(config: `assembly.low_confidence_threshold`), run the plausibility checks.

In [ ]:
def assemble(
    probs: np.ndarray,
    threshold: float = 0.6,
    expected_board: chess.Board | None = None,
) -> BoardPrediction:
    """Turn (64, 13) class probabilities into a BoardPrediction.

    threshold: squares whose top probability is below this are flagged as low-confidence
        (config: assembly.low_confidence_threshold).
    expected_board: optional prior (e.g. the previous position from M5). Start by only
        reporting disagreements in `issues` (informative, not illegal).
    """
    # Design:
    # - Shape check (64, NUM_CLASSES) -> ValueError.
    # - labels = CLASSES[argmax per row], confidence = max per row; labels_to_board.
    # - low_confidence = names of squares below threshold; issues = check_legality;
    #   legal = no legality issues.
    # - expected_board: append "differs from expected board at e2, e4"-style issue.
    # - If not legal, try repair and re-check.
    # Decision for you: how the prior should really be used (e.g. only allow positions one
    # legal move away, which is what M5's infer_move does).
    # TODO: implement
    raise NotImplementedError


def repair(probs: np.ndarray, issues: list[str]) -> np.ndarray | None:
    """Constrained fixing of an illegal board.

    Return the modified (64, 13) probs (or a label list), or None if no legal fix found.
    """
    # Design:
    # - Among low-confidence squares, swap the lowest-confidence one to its second-best class
    #   (or search the k lowest squares x top-2 classes), re-run check_legality.
    # - Keep the most probable legal assignment (sum of log-probs over squares).
    # - Search size grows as 2^k: keep k small (e.g. 4-6).
    # Report how often repair helps in the evaluation.
    # TODO: implement
    raise NotImplementedError

## Self-check
(was tests/test_board_assembly.py)

In [ ]:
def one_hot(board: chess.Board) -> np.ndarray:
    """(64, 13) one-hot probabilities of a board (handy for tests)."""
    # TODO: implement (board_to_labels + CLASS_TO_INDEX)
    raise NotImplementedError


# Self-check plan (write these as asserts once the functions above work):
# - assemble(one_hot(start)) -> fen == chess.STARTING_BOARD_FEN, legal, no issues, no
#   low-confidence squares, labels[e1] == "K", confidence == [1.0] * 64.
# - e4 set to 0.5 empty / 0.5-1e-6 P / 1e-6 p with threshold 0.6 -> low_confidence == ["e4"],
#   label "empty".
# - probs of shape (63, 13) -> ValueError.
# - queen on d1 replaced by a second white king -> not legal, issue "white has 2 kings".
# - black king removed + white pawn on a8 -> "black has 0 kings" and an issue naming a8.
# - extra white pawn on e4 in the start position -> issue containing "9 P".
# - position after 1. e4 with expected_board = start -> still legal, issue names e2 and e4.
# - repair fixes a board where a queen was misread as a second king.
# - status-based checks flag adjacent kings / side not to move in check.
print("board_assembly: self-check not written yet (see plan above)")